# 第28章: 分割表

## 学習目標
- 対数線形モデルを理解する
- 独立性と条件付き独立性を判定できる
- 三元分割表を分析できる
- モデル選択を行える

## 📋 学習メタ情報

### 推定学習時間
**90〜120分**

### 難易度
**★★★☆☆** (5段階中3)

---

## 🎯 なぜこの章を学ぶのか？

この章の内容は、実務での統計的データ分析に直結する重要なトピックです。理論と実践の両面から理解を深めましょう。

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
from itertools import combinations

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
np.random.seed(42)

## 28.1 対数線形モデル

### 二元分割表のモデル
$$\log m_{ij} = \mu + \lambda_i^A + \lambda_j^B + \lambda_{ij}^{AB}$$

- $\mu$: 全体平均
- $\lambda_i^A$: 変数Aの主効果
- $\lambda_j^B$: 変数Bの主効果
- $\lambda_{ij}^{AB}$: 交互作用効果

### 独立モデル
$\lambda_{ij}^{AB} = 0$ のとき独立
$$\log m_{ij} = \mu + \lambda_i^A + \lambda_j^B$$

In [2]:
# 二元分割表の例
np.random.seed(123)

# Observed frequencies
observed = np.array([
    [50, 30, 20],
    [30, 40, 30],
    [20, 30, 50]
])

print("対数線形モデル")
print("="*60)
print("\n観測度数:")
print(observed)

# Row and column totals
row_totals = observed.sum(axis=1)
col_totals = observed.sum(axis=0)
n = observed.sum()

print(f"\n行合計: {row_totals}")
print(f"列合計: {col_totals}")
print(f"総数: {n}")

対数線形モデル

観測度数:
[[50 30 20]
 [30 40 30]
 [20 30 50]]

行合計: [100 100 100]
列合計: [100 100 100]
総数: 300


In [3]:
# 独立性の検定
chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print("独立性の検定")
print("="*60)
print(f"\n期待度数 (独立モデル):")
print(expected.round(1))

print(f"\nカイ二乗統計量: {chi2:.3f}")
print(f"自由度: {dof}")
print(f"p値: {p_value:.4f}")

if p_value < 0.05:
    print("\n→ 独立性を棄却（変数間に関連あり）")
else:
    print("\n→ 独立性を採択（変数間に関連なし）")

独立性の検定

期待度数 (独立モデル):
[[33.3 33.3 33.3]
 [33.3 33.3 33.3]
 [33.3 33.3 33.3]]

カイ二乗統計量: 30.000
自由度: 4
p値: 0.0000

→ 独立性を棄却（変数間に関連あり）


In [4]:
# 対数線形モデルのパラメータ推定
def fit_loglinear_saturated(table):
    """Fit saturated log-linear model and return parameters."""
    I, J = table.shape
    m = table.astype(float)
    log_m = np.log(m)
    
    # Overall mean
    mu = log_m.mean()
    
    # Row effects
    lambda_A = log_m.mean(axis=1) - mu
    
    # Column effects
    lambda_B = log_m.mean(axis=0) - mu
    
    # Interaction effects
    lambda_AB = log_m - mu - lambda_A[:, np.newaxis] - lambda_B
    
    return mu, lambda_A, lambda_B, lambda_AB

mu, lambda_A, lambda_B, lambda_AB = fit_loglinear_saturated(observed)

print("飽和モデルのパラメータ")
print("="*60)
print(f"\n全体平均 μ: {mu:.3f}")
print(f"行効果 λ^A: {lambda_A.round(3)}")
print(f"列効果 λ^B: {lambda_B.round(3)}")
print(f"\n交互作用 λ^AB:")
print(lambda_AB.round(3))

飽和モデルのパラメータ

全体平均 μ: 3.457
行効果 λ^A: [-0.02   0.041 -0.02 ]
列効果 λ^B: [-0.02   0.041 -0.02 ]

交互作用 λ^AB:
[[ 0.496 -0.076 -0.42 ]
 [-0.076  0.151 -0.076]
 [-0.42  -0.076  0.496]]


## 28.2 三元分割表

### モデル階層
1. **完全独立**: $(A)(B)(C)$
2. **条件付き独立**: $(AB)(C)$, $(AC)(B)$, $(BC)(A)$
3. **一様連関**: $(AB)(AC)(BC)$
4. **飽和モデル**: $(ABC)$

### 条件付き独立
$A \perp B | C$: CをfixしたときAとBが独立

In [5]:
# 三元分割表の例
np.random.seed(456)

# 3-way table: A (2) x B (2) x C (2)
# Example: Treatment x Outcome x Gender
table_3way = np.array([
    [[30, 20],   # A=1, C=1
     [15, 35]],  # A=1, C=2
    [[25, 25],   # A=2, C=1
     [20, 30]]   # A=2, C=2
])

labels_A = ['Treatment 1', 'Treatment 2']
labels_B = ['Success', 'Failure']
labels_C = ['Male', 'Female']

print("三元分割表")
print("="*60)

for k, c_label in enumerate(labels_C):
    print(f"\n{c_label}:")
    print(f"{'':>15} {labels_B[0]:>10} {labels_B[1]:>10}")
    for i, a_label in enumerate(labels_A):
        print(f"{a_label:>15} {table_3way[i, 0, k]:>10} {table_3way[i, 1, k]:>10}")

三元分割表

Male:
                   Success    Failure
    Treatment 1         30         15
    Treatment 2         25         20

Female:
                   Success    Failure
    Treatment 1         20         35
    Treatment 2         25         30


In [6]:
# 周辺表の計算
print("周辺表")
print("="*60)

# A x B (C周辺化)
table_AB = table_3way.sum(axis=2)
print("\nA x B (Cを周辺化):")
print(table_AB)

# A x C (B周辺化)
table_AC = table_3way.sum(axis=1)
print("\nA x C (Bを周辺化):")
print(table_AC)

# B x C (A周辺化)
table_BC = table_3way.sum(axis=0)
print("\nB x C (Aを周辺化):")
print(table_BC)

周辺表

A x B (Cを周辺化):
[[50 50]
 [50 50]]

A x C (Bを周辺化):
[[45 55]
 [45 55]]

B x C (Aを周辺化):
[[55 45]
 [35 65]]


In [7]:
# 条件付き独立の検定
print("条件付き独立の検定")
print("="*60)

# Test A ⊥ B | C (Cochran-Mantel-Haenszel)
# For each level of C, compute chi-square
chi2_conditional = 0
for k in range(2):
    table_k = table_3way[:, :, k]
    chi2_k, _, _, _ = stats.chi2_contingency(table_k)
    chi2_conditional += chi2_k
    print(f"\n{labels_C[k]}:")
    print(f"  χ² = {chi2_k:.3f}")

# Marginal test
chi2_marginal, p_marginal, _, _ = stats.chi2_contingency(table_AB)

print(f"\n周辺表 (A x B) のχ²: {chi2_marginal:.3f}")
print(f"条件付きχ²の和: {chi2_conditional:.3f}")

条件付き独立の検定

Male:
  χ² = 0.748

Female:
  χ² = 0.602

周辺表 (A x B) のχ²: 0.000
条件付きχ²の和: 1.350


## 28.3 モデル選択

### 尤度比検定
$$G^2 = 2 \sum_{i,j,k} n_{ijk} \log\left(\frac{n_{ijk}}{\hat{m}_{ijk}}\right)$$

### モデル比較
- 入れ子モデル: $G^2$ の差
- 一般: AIC, BIC

In [8]:
# モデル比較
def compute_G2(observed, expected):
    """Compute likelihood ratio statistic G²."""
    # Avoid log(0)
    obs_flat = observed.flatten()
    exp_flat = expected.flatten()
    mask = obs_flat > 0
    G2 = 2 * np.sum(obs_flat[mask] * np.log(obs_flat[mask] / exp_flat[mask]))
    return G2

def fit_independence_3way(table):
    """Fit complete independence model (A)(B)(C)."""
    n = table.sum()
    p_A = table.sum(axis=(1, 2)) / n
    p_B = table.sum(axis=(0, 2)) / n
    p_C = table.sum(axis=(0, 1)) / n
    
    expected = n * np.outer(p_A, p_B).reshape(2, 2, 1) * p_C
    return expected

def fit_conditional_independence(table, given='C'):
    """Fit conditional independence model."""
    n = table.sum()
    
    if given == 'C':  # (AC)(BC)
        expected = np.zeros_like(table, dtype=float)
        for k in range(2):
            n_k = table[:, :, k].sum()
            p_A_k = table[:, :, k].sum(axis=1) / n_k
            p_B_k = table[:, :, k].sum(axis=0) / n_k
            expected[:, :, k] = n_k * np.outer(p_A_k, p_B_k)
    
    return expected

print("モデル選択")
print("="*60)

# Complete independence
exp_indep = fit_independence_3way(table_3way)
G2_indep = compute_G2(table_3way, exp_indep)
df_indep = 7 - 3  # 8 cells - 4 parameters

# Conditional independence A ⊥ B | C
exp_cond = fit_conditional_independence(table_3way, 'C')
G2_cond = compute_G2(table_3way, exp_cond)
df_cond = 7 - 5  # 8 cells - 6 parameters

print(f"\n{'モデル':^30} {'G²':>8} {'df':>5} {'p値':>10}")
print("-"*55)

p_indep = 1 - stats.chi2.cdf(G2_indep, df_indep)
p_cond = 1 - stats.chi2.cdf(G2_cond, df_cond)

print(f"{'完全独立 (A)(B)(C)':^30} {G2_indep:>8.3f} {df_indep:>5} {p_indep:>10.4f}")
print(f"{'条件付き独立 (AC)(BC)':^30} {G2_cond:>8.3f} {df_cond:>5} {p_cond:>10.4f}")

モデル選択

             モデル                     G²    df         p値
-------------------------------------------------------
        完全独立 (A)(B)(C)           10.252     4     0.0364
       条件付き独立 (AC)(BC)            2.114     2     0.3476


In [9]:
# AICとBICによる比較
n = table_3way.sum()

# Number of parameters for each model
n_params = {'完全独立': 4, '条件付き独立': 6, '飽和': 8}

# Log-likelihood (up to constant)
def log_likelihood(observed, expected):
    obs_flat = observed.flatten()
    exp_flat = expected.flatten()
    mask = obs_flat > 0
    return np.sum(obs_flat[mask] * np.log(exp_flat[mask]))

ll_indep = log_likelihood(table_3way, exp_indep)
ll_cond = log_likelihood(table_3way, exp_cond)
ll_sat = log_likelihood(table_3way, table_3way.astype(float))

print("\n情報量基準")
print("="*60)
print(f"\n{'モデル':^30} {'AIC':>10} {'BIC':>10}")
print("-"*55)

for name, ll, k in [('完全独立', ll_indep, 4), 
                     ('条件付き独立', ll_cond, 6), 
                     ('飽和', ll_sat, 8)]:
    aic = -2 * ll + 2 * k
    bic = -2 * ll + k * np.log(n)
    print(f"{name:^30} {aic:>10.2f} {bic:>10.2f}")


情報量基準

             モデル                      AIC        BIC
-------------------------------------------------------
             完全独立                -1281.55   -1268.36
            条件付き独立               -1285.69   -1265.90
              飽和                 -1283.81   -1257.42


## 28.4 オッズ比

### 定義
$$\text{OR} = \frac{n_{11} n_{22}}{n_{12} n_{21}}$$

### 対数オッズ比
$$\log(\text{OR}) = \lambda_{11}^{AB} - \lambda_{12}^{AB} - \lambda_{21}^{AB} + \lambda_{22}^{AB}$$

In [10]:
# オッズ比の計算
print("オッズ比")
print("="*60)

# For each level of C
for k, c_label in enumerate(labels_C):
    table_k = table_3way[:, :, k]
    OR = (table_k[0, 0] * table_k[1, 1]) / (table_k[0, 1] * table_k[1, 0])
    log_OR = np.log(OR)
    
    # Standard error of log OR
    se_log_OR = np.sqrt(1/table_k[0, 0] + 1/table_k[0, 1] + 
                        1/table_k[1, 0] + 1/table_k[1, 1])
    
    # 95% CI
    ci_low = np.exp(log_OR - 1.96 * se_log_OR)
    ci_high = np.exp(log_OR + 1.96 * se_log_OR)
    
    print(f"\n{c_label}:")
    print(f"  オッズ比: {OR:.3f}")
    print(f"  対数オッズ比: {log_OR:.3f}")
    print(f"  95% CI: [{ci_low:.3f}, {ci_high:.3f}]")

オッズ比

Male:
  オッズ比: 1.600
  対数オッズ比: 0.470
  95% CI: [0.681, 3.760]

Female:
  オッズ比: 0.686
  対数オッズ比: -0.377
  95% CI: [0.319, 1.472]


## 28.5 練習問題

### 問題1
対数線形モデルにおいて飽和モデルとはどのようなモデルか。

### 問題2
条件付き独立と周辺独立の違いを例を用いて説明せよ。

### 問題3
以下の2×2分割表のオッズ比を計算せよ。
$$\begin{pmatrix} 40 & 10 \\ 20 & 30 \end{pmatrix}$$

In [11]:
# 問題1の解答
print("問題1: 飽和モデル")
print("="*60)

print("""
【飽和モデルとは】
すべての効果（主効果と交互作用）を含むモデル

【二元分割表の場合】
log m_ij = μ + λᵢᴬ + λⱼᴮ + λᵢⱼᴬᴮ

【三元分割表の場合】
log m_ijk = μ + λᵢᴬ + λⱼᴮ + λₖᶜ + λᵢⱼᴬᴮ + λᵢₖᴬᶜ + λⱼₖᴮᶜ + λᵢⱼₖᴬᴮᶜ

【特徴】
1. パラメータ数 = セル数
2. 自由度 = 0
3. 期待度数 = 観測度数（完全適合）
4. G² = 0

【使い方】
・他のモデルとの比較の基準
・交互作用の大きさを調べる
""")

問題1: 飽和モデル

【飽和モデルとは】
すべての効果（主効果と交互作用）を含むモデル

【二元分割表の場合】
log m_ij = μ + λᵢᴬ + λⱼᴮ + λᵢⱼᴬᴮ

【三元分割表の場合】
log m_ijk = μ + λᵢᴬ + λⱼᴮ + λₖᶜ + λᵢⱼᴬᴮ + λᵢₖᴬᶜ + λⱼₖᴮᶜ + λᵢⱼₖᴬᴮᶜ

【特徴】
1. パラメータ数 = セル数
2. 自由度 = 0
3. 期待度数 = 観測度数（完全適合）
4. G² = 0

【使い方】
・他のモデルとの比較の基準
・交互作用の大きさを調べる



In [12]:
# 問題2の解答
print("問題2: 条件付き独立と周辺独立")
print("="*60)

print("""
【周辺独立】
P(A, B) = P(A)P(B)
→ AとBが無条件で独立

【条件付き独立】
P(A, B | C) = P(A | C)P(B | C)
→ Cを固定するとAとBが独立

【違いの例（シンプソンのパラドックス）】
治療 (A) と回復 (B) と性別 (C) の関係:

周辺表では治療と回復に関連がある（非独立）
しかし各性別内では独立（条件付き独立）

→ 交絡変数（性別）を制御することで
  真の関係が見える

【数学的関係】
条件付き独立 ≠> 周辺独立
周辺独立 ≠> 条件付き独立
""")

問題2: 条件付き独立と周辺独立

【周辺独立】
P(A, B) = P(A)P(B)
→ AとBが無条件で独立

【条件付き独立】
P(A, B | C) = P(A | C)P(B | C)
→ Cを固定するとAとBが独立

【違いの例（シンプソンのパラドックス）】
治療 (A) と回復 (B) と性別 (C) の関係:

周辺表では治療と回復に関連がある（非独立）
しかし各性別内では独立（条件付き独立）

→ 交絡変数（性別）を制御することで
  真の関係が見える

【数学的関係】
条件付き独立 ≠> 周辺独立
周辺独立 ≠> 条件付き独立



In [13]:
# 問題3の解答
print("問題3: オッズ比の計算")
print("="*60)

table = np.array([[40, 10], [20, 30]])

print("\n分割表:")
print(table)

OR = (table[0, 0] * table[1, 1]) / (table[0, 1] * table[1, 0])
log_OR = np.log(OR)

print(f"\nオッズ比 = (40 × 30) / (10 × 20) = {OR:.1f}")
print(f"対数オッズ比 = ln({OR:.1f}) = {log_OR:.3f}")

# Interpretation
print(f"\n解釈:")
print(f"行1のオッズは行2のオッズの{OR:.1f}倍")

問題3: オッズ比の計算

分割表:
[[40 10]
 [20 30]]

オッズ比 = (40 × 30) / (10 × 20) = 6.0
対数オッズ比 = ln(6.0) = 1.792

解釈:
行1のオッズは行2のオッズの6.0倍


## ⚠️ よくある間違いと解決策

統計分析では、手法の前提条件を確認せずに適用してしまうことがよくあります。必ず前提を確認し、適切な手法を選択しましょう。

---

## 📝 理解度チェック

この章で学んだ内容を振り返り、重要な概念を自分の言葉で説明できるか確認しましょう。

---

## 📚 まとめ

お疲れ様でした！この章で学んだ手法は、実際のデータ分析で頻繁に使われます。実データで試して理解を深めましょう。

---